# Multimodal Recording example
Load all bundled signals into one `Recording`, assign shared temporal segments, and process each modality independently.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from multimodalphysiokit.io import read_biosignalsplux_hdf5
from multimodalphysiokit.processors import (
    ECGProcessor,
    EDAProcessor,
    FNIRSProcessor,
    RespirationProcessor,
    TemperatureProcessor,
)
from multimodalphysiokit.utils import plot_signal_with_phases

data_file = Path("examples/sample_data/synthetic_multimodal_180s.hdf5")
recording = read_biosignalsplux_hdf5(data_file)
recording.signal_names()

In [ ]:
recording.set_phases(
    np.array([[0.0, 90.0], [90.0, 180.0]]),
    ["segment_1", "segment_2"],
)
{name: recording.phase_indices(name).tolist() for name in recording.signal_names()}

In [ ]:
plot_signal_with_phases(recording, "ecg", title="Raw ECG signal", show=True)
plot_signal_with_phases(recording, "eda", title="Raw EDA signal", show=True)
plot_signal_with_phases(recording, "rip", title="Raw respiration signal", show=True)
plot_signal_with_phases(recording, "temp", title="Raw temperature signal", show=True)
plot_signal_with_phases(recording, "fnirs_red", title="Raw fNIRS red signal", show=True)
plot_signal_with_phases(recording, "fnirs_infrared", title="Raw fNIRS infrared signal", show=True)

In [ ]:
results = {
    "ecg": recording.process_phases(ECGProcessor(label_frequency_analysis=1), "ecg"),
    "respiration": recording.process_phases(RespirationProcessor(), "rip"),
    "temperature": recording.process_phases(TemperatureProcessor(), "temp"),
}

In [ ]:
fnirs_processor = FNIRSProcessor(age_years=35)
baseline_values = fnirs_processor.compute_baseline_values(
    recording.extract_phase("fnirs_red", "segment_1"),
    recording.extract_phase("fnirs_infrared", "segment_1"),
)
results["fnirs"] = recording.process_phases(
    fnirs_processor,
    ("fnirs_red", "fnirs_infrared"),
    process_kwargs={
        "baseline_red_value": baseline_values[0],
        "baseline_infrared_value": baseline_values[1],
    },
)

In [ ]:
try:
    results["eda"] = recording.process_phases(EDAProcessor(), "eda")
except ImportError as error:
    print(f"Optional EDA processing skipped: {error}")

{
    modality: {phase: len(result.features) for phase, result in phase_results.items()}
    for modality, phase_results in results.items()
}

In [ ]:
result_rows = [
    {
        "modality": modality,
        "phase": phase_label,
        "feature_name": feature_name,
        "value": feature_value.item() if isinstance(feature_value, np.generic) else feature_value,
    }
    for modality, phase_results in results.items()
    for phase_label, processing_result in phase_results.items()
    for feature_name, feature_value in processing_result.features.items()
]

results_table = pd.DataFrame(
    result_rows,
    columns=["modality", "phase", "feature_name", "value"],
)
display(results_table.assign(value=results_table["value"].round(6)))